# M0: установка и CPU preflight

Укажите свой Git URL и полный commit SHA. Notebook устанавливает выбранный код и вызывает тот же CLI, что используется локально. Сначала проверьте выбранный репозиторий и commit. Ответы кандидатов не исполняются: demo использует ScriptedPolicy и FakeExecutor, а результаты имеют `synthetic_demo=true`.

Для M0 достаточно CPU runtime. Подключение Qwen, GPU и настоящего executor относится к следующему этапу в `docs/next_steps.md`. В этом notebook нет обучения.


In [ ]:
REPO_URL = ""  # HTTPS URL репозитория без токена доступа.
COMMIT_SHA = ""  # Полный SHA выбранного commit (40 hex-символов).
CHECKOUT_DIR = "/content/qwen-repair"
CONFIG_PATH = "configs/local_demo.yaml"


In [ ]:
import re
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlparse

repo = urlparse(REPO_URL)
if repo.scheme != "https" or not repo.hostname or repo.username or repo.password:
    raise ValueError("Укажите HTTPS URL репозитория без встроенных учётных данных.")
if re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT_SHA) is None:
    raise ValueError("Укажите полный commit SHA из 40 hex-символов.")
checkout = Path(CHECKOUT_DIR)
if checkout.exists():
    raise FileExistsError("Выберите новый CHECKOUT_DIR, чтобы сохранить существующий checkout.")
subprocess.run(["git", "clone", "--", REPO_URL, str(checkout)], check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=checkout, check=True)
actual_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
if actual_sha.lower() != COMMIT_SHA.lower():
    raise RuntimeError("Checkout не соответствует выбранному commit.")
subprocess.run([sys.executable, "-m", "pip", "install", "."], cwd=checkout, check=True)


In [ ]:
subprocess.run(
    [sys.executable, "-m", "qwen_repair", "doctor", "--config", CONFIG_PATH],
    cwd=checkout,
    check=True,
)


In [ ]:
from uuid import uuid4

run_dir = checkout / "runs" / f"colab-m0-{uuid4().hex}"
subprocess.run(
    [sys.executable, "-m", "qwen_repair", "demo", "--config", CONFIG_PATH, "--output", str(run_dir)],
    cwd=checkout,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "qwen_repair", "summarize", "--run", str(run_dir)],
    cwd=checkout,
    check=True,
)
print(f"Артефакты: {run_dir}")


Скачайте созданный каталог запуска до завершения runtime. `metrics.json` описывает синтетические сценарии, а не качество модели. M1 начинается отдельным заданием: выбор GPU, проверка изоляции, inference Qwen и короткие проверки обучения с resume.
